# Análisis Exploratorio Básico - Sesiones de Diputados

Análisis exploratorio del dataset de versiones taquigráficas de la Cámara de Diputados de la Nación Argentina.

## 0. Setup

In [ ]:
import glob
import re
import os
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

plt.rcParams.update({
    "figure.figsize": (12, 5),
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.labelsize": 11,
    "font.family": "sans-serif",
})

OUTPUT_DIR = "figuras"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Cargar datos

In [ ]:
# Ajustá esta ruta si tus parquets están en otro lado
archivos = sorted(glob.glob("../data/parquets/*.parquet"))
print(f"Archivos encontrados: {len(archivos)}")

df = pd.concat([pd.read_parquet(f) for f in archivos], ignore_index=True)
df_texto = df[df["texto_pdf"].notna()].copy()

print(f"Total sesiones: {len(df)}")
print(f"Con texto: {len(df_texto)}")
print(f"Sin texto: {df['texto_pdf'].isna().sum()}")
df_texto.head()

## 2. Cantidad de palabras por sesión

In [ ]:
df_texto["n_palabras"] = df_texto["texto_pdf"].apply(lambda x: len(x.split()))

print(f"Promedio:  {df_texto['n_palabras'].mean():,.0f} palabras por sesión")
print(f"Mediana:   {df_texto['n_palabras'].median():,.0f} palabras por sesión")
print(f"Mínimo:    {df_texto['n_palabras'].min():,.0f} palabras")
print(f"Máximo:    {df_texto['n_palabras'].max():,.0f} palabras")
print(f"Total:     {df_texto['n_palabras'].sum():,.0f} palabras en todo el corpus")

In [ ]:
fig, ax = plt.subplots()
ax.hist(df_texto["n_palabras"], bins=40, color="#4A90D9", edgecolor="white", alpha=0.85)
ax.set_title("Distribución de cantidad de palabras por sesión")
ax.set_xlabel("Cantidad de palabras")
ax.set_ylabel("Cantidad de sesiones")
ax.axvline(df_texto["n_palabras"].median(), color="#E74C3C", linestyle="--",
           label=f"Mediana: {df_texto['n_palabras'].median():,.0f}")
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/hist_palabras_por_sesion.png")
plt.show()

## 3. Sesiones por período

In [ ]:
sesiones_por_periodo = df_texto.groupby("id_periodo").size().sort_index()

fig, ax = plt.subplots()
ax.bar(sesiones_por_periodo.index.astype(str), sesiones_por_periodo.values,
       color="#4A90D9", edgecolor="white")
ax.set_title("Cantidad de sesiones con texto por período parlamentario")
ax.set_xlabel("Período")
ax.set_ylabel("Sesiones")
ax.tick_params(axis="x", rotation=90, labelsize=8)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/sesiones_por_periodo.png")
plt.show()

## 4. Sesiones por año

In [ ]:
df_texto["anio"] = pd.to_datetime(df_texto["fecha"]).dt.year
sesiones_por_anio = df_texto.groupby("anio").size()

fig, ax = plt.subplots()
ax.plot(sesiones_por_anio.index, sesiones_por_anio.values,
        marker="o", color="#4A90D9", linewidth=2, markersize=5)
ax.set_title("Cantidad de sesiones por año")
ax.set_xlabel("Año")
ax.set_ylabel("Sesiones")
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/sesiones_por_anio.png")
plt.show()

## 5. Palabras más frecuentes

In [ ]:
# Stopwords del español + palabras del formato parlamentario
STOPWORDS = {
    # Español comunes
    "de", "la", "el", "en", "y", "que", "los", "del", "las", "un", "por",
    "con", "una", "para", "es", "se", "no", "al", "lo", "su", "más", "como",
    "pero", "sus", "le", "ha", "me", "si", "sin", "sobre", "este", "ya",
    "entre", "también", "fue", "ser", "son", "dos", "tiene", "está", "desde",
    "todo", "esta", "nos", "ni", "hay", "muy", "ese", "esa", "esos", "esas",
    "uno", "otra", "otro", "estos", "estas", "o", "a", "e", "i", "u",
    "mi", "te", "tu", "él", "ella", "ellos", "ellas", "hemos", "han",
    "era", "sido", "tiene", "puede", "todos", "todas", "cada", "donde",
    "cuando", "aquí", "así", "después", "antes", "bien", "hacer", "porque",
    "hasta", "yo", "usted", "ustedes", "nosotros", "vamos", "tiene",
    "decir", "dice", "dijo", "creo", "quiero", "voy", "van", "va",
    "están", "estamos", "esto", "eso", "cual", "cuál", "qué",
    # Formato parlamentario
    "señor", "señora", "diputado", "diputada", "presidente", "presidenta",
    "secretario", "secretaria", "cámara", "sesión", "reunión",
    "artículo", "proyecto", "ley", "orden", "día", "comisión",
    "honorable", "nación", "república", "argentina", "buenos", "aires",
    "página", "pág", "núm", "número", "inc", "art",
    "sr", "sra", "dr", "dra",
}

In [ ]:
print("Procesando corpus completo...")
todo_el_texto = " ".join(df_texto["texto_pdf"].values)
palabras = re.findall(r"[a-záéíóúñü]+", todo_el_texto.lower())
palabras_filtradas = [p for p in palabras if p not in STOPWORDS and len(p) > 3]

conteo = Counter(palabras_filtradas)
top_30 = conteo.most_common(30)

print("\nTop 30 palabras más frecuentes:")
for i, (palabra, freq) in enumerate(top_30, 1):
    print(f"  {i:2d}. {palabra:<20s} {freq:>10,}")

In [ ]:
palabras_top, frecuencias_top = zip(*reversed(top_30))

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(palabras_top, frecuencias_top, color="#4A90D9", edgecolor="white")
ax.set_title("Top 30 palabras más frecuentes (sin stopwords)")
ax.set_xlabel("Frecuencia")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/top_30_palabras.png")
plt.show()

## 6. Evolución del largo de sesiones en el tiempo

In [ ]:
promedio_por_anio = df_texto.groupby("anio")["n_palabras"].mean()

fig, ax = plt.subplots()
ax.plot(promedio_por_anio.index, promedio_por_anio.values,
        marker="o", color="#E74C3C", linewidth=2, markersize=5)
ax.set_title("Promedio de palabras por sesión según año")
ax.set_xlabel("Año")
ax.set_ylabel("Promedio de palabras")
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/promedio_palabras_por_anio.png")
plt.show()

## 7. Tipos de sesión

In [ ]:
def extraer_tipo(desc):
    desc = desc.lower()
    if "expresiones en minoría" in desc:
        return "Expresiones en Minoría"
    elif "asamblea legislativa" in desc:
        return "Asamblea Legislativa"
    elif "preparatoria" in desc:
        return "Sesión Preparatoria"
    elif "informativa" in desc:
        return "Sesión Informativa"
    elif "especial" in desc:
        return "Sesión Especial"
    elif "extraordinaria" in desc:
        return "Sesión Extraordinaria"
    elif "ordinaria" in desc:
        return "Sesión Ordinaria"
    elif "homenaje" in desc:
        return "Sesión de Homenaje"
    elif "no efectuada" in desc or "fracasada" in desc:
        return "No efectuada"
    else:
        return "Otro"

df_texto["tipo_sesion"] = df_texto["descripcion"].apply(extraer_tipo)
tipos = df_texto["tipo_sesion"].value_counts()
print(tipos.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
tipos.plot(kind="barh", ax=ax, color="#4A90D9", edgecolor="white")
ax.set_title("Distribución por tipo de sesión")
ax.set_xlabel("Cantidad")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/tipos_sesion.png")
plt.show()

## 8. Resumen

In [ ]:
print("RESUMEN DEL DATASET")
print("=" * 40)
print(f"Períodos:              {df['id_periodo'].nunique()}")
print(f"Sesiones totales:      {len(df)}")
print(f"Sesiones con texto:    {len(df_texto)}")
print(f"Palabras totales:      {df_texto['n_palabras'].sum():,}")
print(f"Promedio por sesión:   {df_texto['n_palabras'].mean():,.0f}")
print(f"Rango de años:         {df_texto['anio'].min()} - {df_texto['anio'].max()}")